# Sequential Counter For Ladder encoding

### I. Ý tưởng

**SCL thực chất là:**
- Dùng Sequential counter để encode prefix sum = 1, rồi tái sử dụng giữa các cửa sổ chồng lấp
- Nói cách khác:
  + Không encode từng AMO riêng lẻ
  + Không encode toàn cục
  + encode từng block + kết nối các block đó

### II. Chi tiết

**1. SCAMO**
$\mathrm{SCAMO}(\Omega, w) = \bigwedge_{i=-w}^{n} \left( \sum_{j=i}^{i+w+1} x_j \le 1 \right)$ \
$X_1 + X_2 + X_3+ X_4 \le 1$ \
$\ \ \ \ \ \ \ \ \ $ $X_2 + X_3+ X_4 + X_5 \le 1$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ $ $X_3 + X_4 + X_5 + X_6 \le 1$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ $ $X_4 + X_5 + X_6 + X_7 \le 1$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  \ $ $X_5 + X_6 + X_7 + X_8 \le 1$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  \ \ \ \ \ \ \ \ \ \  \ $ $X_6 + X_7 + X_8 + X_9 \le 1$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  \ \ \ \ \ \ \ \  \ \ \ \ \ \ \ \ \ \ \ \ \ $ $X_7 + X_8 + X_9 + X_{10} \le 1$

**2. Encoding**

**a. Cách chia Block** 

<img src="D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock.jpg" width="500"/>

- Nguyên tắc phân chia
    + Hình dáng: Hình dung như một hình bình hành được chia thành nhiều phần theo chiều ngang.
    + Số khối chính: $M = \left\lceil \frac{n}{w} \right\rceil$ ($w$ là số phần tử trên 1 hàng)
    + Kích thước: Mỗi khối chứa tối đa $w$ phần tử ($w$ là số phần tử trên 1 hàng, $n$ là tổng số biến chính).
    + Khối cuối: Có thể chứa ít phần tử hơn nếu tổng số phần tử không chia hết cho $w$.
    + Liên kết: Các khối không độc lập hoàn toàn mà liên kết qua các ràng buộc (như AMZ) để duy trì tính chất toàn cục.
  | Khối (Area) | Các phần tử           | Ghi chú                     |
|--------------|----------------------|-----------------------------|
| Khối 1       | $x_1, x_2, x_3, x_4$ | Đủ $w$ phần tử              |
| Khối 2       | $x_5, x_6, x_7, x_8$ | Đủ $w$ phần tử              |
| Khối 3       | $x_9, x_{10}$        | Khối cuối (còn lại 2 phần tử) |

* **Theo tôi** (`không nói rõ trong paper `Sequential Counter Encoding for Staircase At-Most-One Constraints)
    + Khi chia thành 3 khối thì `khối chung gian` (khối 2) lại không thể encoding bằng SC nên ta phải cắt đôi khối 2 ra để từ 1 hình `bình hành` thành 2 hình `tam giác`
    + Khi cắt khối trung gian thành 2 khối, ta để `hàng cuối của tam giác 1` là `hàng đầu của tam giác 2`. Đơn giản là để các khối được chia đều nhau dễ code, tối ưu có thể loại bỏ vùng overlap đó (nghĩ thế thôi :>>)
      > quy ước trong bài biết viết này các khối được cắt theo hàng ngang gọi là Area, sau khi cắt các Area thành các tam giác thì gọi là Block

<img src= "D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock2.png" width="700"/>

> Ảnh trên có bổ xung thêm biến $X_8$ ở cuối để encode 

**b. Các công thức Encode AMO (xét trong 1 block)**
+ $X_i \rightarrow R_i (0 < i < n)$ 
+ $R_{i-1} \rightarrow R_i (1 < i < n)$ 
+ $\neg R_i \land \neg R_{i-1} \rightarrow \neg R_i  (1 < i < n)$ 
+  $R_{i-1} \rightarrow \neg X_i (1 < i \le n)$

**c. Cách nối các Block**

- Ý tưởng: Nếu Block này đã có bit $1$ thì khối kia không được có bit $1$
- Chú thích: $R_{i,j}$ để kiểm tra xem chuỗi có $j$ phần tử ở block $i$ đã có ít nhất $1$ biến $True$ hay chưa
  + $R_{i,j} \leftrightarrow \sum X \ge 1$ 
  + $R_{i,j} \leftrightarrow \sum X \le 0$
    

$X_1 + X_2 + X_3+ X_4 \equiv R_{1, 4}$ \
$\ \ \ \ \ \ \ \ \ $ $X_2 + X_3+ X_4 \equiv R_{1, 3}$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ $ $X_3 + X_4 \equiv R_{1, 2}$ \
$\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ $ $X_4 \equiv R_{1, 1}$  \
Khối $B_1$

$X_5 \equiv R_{2,1}$ \
$X_5 + X_6 \equiv R_{2,2}$ \
$X_5 + X_6 + X_7 \equiv R_{2,3}$ \
$X_5 + X_6 + X_7 + X_8 \equiv R_{2,4}$ \
Khối $B_2$


- Nối:
    + Dòng 1: $X_1+ X_2 + X_3 + X_4 \le 1$
    + Dòng 2: 
        * Tổng lớn `nhỏ hơn hoặc bằng 1` thì tổng nhỏ phải `nhỏ hơn hoặc bằng 1` \
            $X_2 + X_3 + X_4 + X_5 \le 1 $ \
            $\rightarrow (X_2 + X_3 + X_4 \le 1) \land (X_5 \le 1)$
        * Nhưng phải `loại` bỏ trường hợp `cả hai tổng cùng bằng 1` nghĩa là một trong hai phải bằng 0 \
            $(X_2 + X_3 + X_4 + X_5 \le 0) \lor (X_5 \le 0) \equiv \neg R_{1,3} \lor \neg X_5$
        *  Vậy nối dòng 1 cần mệnh đề: \
            $ (X_2 + X_3 + X_4 \le 1) \land (X_5 \le 1) \land (\neg R_{1,3} \lor \neg X_5)$
    + Các dòng khác tương tự

**3. Cách chia block trên có tối ưu hay không? Khong sequential counter encoding không sử dụng công thức $\neg R_i \land \neg R_{i-1} \rightarrow \neg R_i  (1 < i < n)$  vậy tại sao khi encoding từng block lại sử dụng?**

a. Để trả lời ý đầu tiên ta xem xét còn cách chia nào khác không?

<img src="D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock2.jpg" width="400"/><img src="D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock3.jpg" width="400"/>

<img src="D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock4.jpg" width="400"/><img src="D:\SAT\SAT_TranDucThanh_24021630\image\chiaBlock5.jpg" width="400"/>

Đoán thôi có vẻ nếu cách chia càng lệch thì càng nhiều mệnh đề, tối ưu ta có thể loại bỏ vùng overlap như hình 1

b. Chịu

**4. Code**

**Lưu ý:**
- `Area` là các vùng được chia theo hàng ngang, mỗi vùng chứa $w$ biến
- Trong mỗi vùng lại có 2 `Block` trừ `Area` đầu và cuối
- Kết nối các `Block` bản chất cũng như kết nối các `Area`, `Block` cuối của `Area` này sẽ nối với `Block` đầu của `Area` kia
- Số thứ tự của `Area` và `Block` bắt đầu từ 0

In [2]:
from pysat.solvers import Glucose3
import math

**a. At most one của từng Block**

In [26]:
def AMO(g, vars, start_id_var):
    n = len(vars)
    temp_var = list(range(start_id_var, start_id_var + n))
    for j in range(n - 1):
        g.add_clause([-vars[j], temp_var[j]])
    for j in range(1, n - 1):
        g.add_clause([-temp_var[j - 1], temp_var[j]])
    for j in range(1, n - 1):
        g.add_clause([-vars[j], -temp_var[j - 1], temp_var[j]])
    for j in range(1, n):
        g.add_clause([ -vars[j], -temp_var[j - 1]])
    return start_id_var + n 

**b. Xác định Area thứ number có những phần tử nào**

In [4]:
def Area(vars, weight, number):
    
    start_index = weight * number
    end_index = min(start_index + weight, len(vars))

    return vars[start_index:end_index]

**c. Encoding cho từng Area**

In [5]:
def EncodingAreas(g, area, number, n_Areas, start_id_var):
    next_var = start_id_var
    if(number != 0):
        next_var = AMO(g, area, start_id_var)
    if(number != n_Areas - 1):
        next_var = AMO(g, area[::-1], next_var)
    return next_var 

**d. Nối các Area**

In [6]:
def ConnectAreas(g,  n_vars, area1, area2, number1, number2, weight):    
    number_of_block1 = NumberBlock(number1, False, weight)
    number_of_block2 = NumberBlock(number2, True, weight)

    start_var_block1 = StartVarBlock(n_vars, number_of_block1, weight)
    start_var_block2 = StartVarBlock(n_vars, number_of_block2, weight)
    overlap = weight - 1

    for i in range(overlap):

        left_var  = start_var_block1 + i
        right_var = start_var_block2 + (overlap - 1 - i)

        g.add_clause([-left_var, -right_var])

**e. Sequential Contrain bao gồm Encoding từng Area và nối các Area lại**

In [7]:
def SC(g, vars, weight):
    n_var = len(vars)
    next_var = n_var + 1
    n_Areas = math.ceil(len(vars) / weight)

    for number in range(n_Areas):
        area = Area(vars, weight, number)
        next_var = EncodingAreas(g, area, number, n_Areas, next_var)

    for number in range(n_Areas - 1):
        area1 = Area(vars, weight, number)
        area2 = Area(vars, weight, number + 1)
        ConnectAreas(g, n_var, area1, area2, number, number + 1, weight)

**f. Xác định số thứ tự của 1 Block khi biết số thứ tự của Area tương ứng**

In [8]:
def NumberBlock(number_of_area, isFirst, weight):
    number_of_block = 2 * number_of_area
    if isFirst:
        number_of_block -= 1
    return number_of_block

**g. Xác định biến phụ đầu tiên của 1 Block khi biết số thứ tự của Block đó**

In [9]:
def StartVarBlock(n_vars, number_of_block, weight):
    return n_vars + number_of_block * weight + 1

**h. Main để kiểm tra**

In [25]:
def main():

    g = Glucose3()

    vars = list(range(1, 11))
    weight = 4
    SC(g, vars, weight)

    #gán 1 số biến true để kiểm tra 
    #g.add_clause([4])
    g.add_clause([2])

    
    result = g.solve()

    print("SAT:", result)

    if result:

        model = g.get_model()
        true_vars = [x for x in model if x > 0]
        print("Biến nhận giá trị 1:")
        print(true_vars)

if __name__ == "__main__":
    main()

SAT: True
Biến nhận giá trị 1:
[2, 13]
